In [ ]:
# learning from the blog : https://victorzhou.com/

# Part 1: CNN with forward pass

In [2]:
import numpy as np

class Conv3x3:
    # A convolutional layer using 3x3 filter
    def __init__(self, num_filters):
        self.num_filters = num_filters
        # filters is a 3d array with dimensions (num_filters, 3, 3)
        # We divide by 9 to reduce the variance of our initial values
        self.filters = np.random.randn(num_filters, 3,3) / 9
    
    def iterate_regions(self, image):
        # Generate all possible 3x3 image regions using valid padding -image is 2d numpy array
        h, w = image.shape
        for i in range(h-2):
            for j in range(w-2):
                im_region = image[i: (i+3), j: (j+3)]
                yield im_region, i, j
    
    def forward(self, input):
        # performs a forward pass of the conv layer using the given input
        # return a 3d numpy arrray with dimensions (h, w, num_filters)
        # input is 2d numpy array
        
        h, w = input.shape
        output = np.zeros((h-2, w-2, self.num_filters))
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.sum(im_region * self.filters, axis= (1, 2))
        return output
        

In [3]:
!pip install mnist==0.2.2

In [5]:
import mnist
# The mnist package handles the MNIST dataset for us!
# Learn more at https://github.com/datapythonista/mnist
train_images = mnist.train_images()
train_labels = mnist.train_labels()

conv = Conv3x3(8)
output = conv.forward(train_images[0])
print(output.shape) # (26, 26, 8)

(26, 26, 8)


In [3]:
import numpy as np

class MaxPool2:
    # max pooling using a pool size of 2
    
    def iterate_regions(self, image):
        # Generates non-overlapping 2x2 image region to pool over
        # image is a 2d numpy array
        h, w, _ = image.shape
        new_h = h // 2
        new_w = w // 2
        
        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2): (i*2 +2), (j*2):(j*2+2)]
                yield im_region, i, j
                
    def forward(self, input):
        # performs forward pass of the maxpooling layer using the given input
        # returns a 3d numpy array iwth dimensions (h/2, w/2, num_filters).
        # input is a 3d numpy array with dimensions (h, w, num_filters)
        h, w, num_filters = input.shape
        output = np.zeros((h //2, w//2, num_filters))
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.amax(im_region, axis= (0,1))
            
        return output

In [10]:
import mnist
train_images = mnist.train_images()
train_labels = mnist.train_labels()

conv = Conv3x3(8)
pool = MaxPool2()

output = conv.forward(train_images[0])
output = pool.forward(output)
print(output.shape) # (13, 13, 8)

(13, 13, 8)


In [13]:
import numpy as np

class Softmax:
    
    # A standard fully connected layer with softmax activation
    
    def __init__(self, input_len, nodes):
        # we divide by input_len to reduce the variance of our initial values
        self.weights = np.random.randn(input_len, nodes) / input_len
        self.biases = np.zeros(nodes)
        
    def forward(self, input):
        # performs a forward pass of softmax layer using the given input.
        # return a 1d numpy array containing the respective probabilities
        input = input.flatten()
        input_len, nodes = self.weights.shape
        totals = np.dot(input, self.weights) + self.biases
        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)

In [9]:
# Putting all together
import mnist
import numpy as np

# We only use the first 1k testing examples (out of 10k total)
# in the interest of time. Feel free to change this if you want.
test_images = mnist.test_images()[:1000]
test_labels = mnist.test_labels()[:1000]

conv = Conv3x3(8)                  # 28x28x1 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13 * 13 * 8, 10) # 13x13x8 -> 10

def forward(image, label):
  '''
  Completes a forward pass of the CNN and calculates the accuracy and
  cross-entropy loss.
  - image is a 2d numpy array
  - label is a digit
  '''
  # We transform the image from [0, 255] to [-0.5, 0.5] to make it easier
  # to work with. This is standard practice.
  out = conv.forward((image / 255) - 0.5)
  out = pool.forward(out)
  out = softmax.forward(out)

  # Calculate cross-entropy loss and accuracy. np.log() is the natural log.
  loss = -np.log(out[label])
  acc = 1 if np.argmax(out) == label else 0

  return out, loss, acc

print('MNIST CNN initialized!')

loss = 0
num_correct = 0
for i, (im, label) in enumerate(zip(test_images, test_labels)):
  # Do a forward pass.
  _, l, acc = forward(im, label)
  loss += l
  num_correct += acc

  # Print stats every 100 steps.
  if i % 100 == 99:
    print(
      '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
      (i + 1, loss / 100, num_correct)
    )
    loss = 0
    num_correct = 0

MNIST CNN initialized!
[Step 100] Past 100 steps: Average Loss 2.302 | Accuracy: 4%
[Step 200] Past 100 steps: Average Loss 2.302 | Accuracy: 9%
[Step 300] Past 100 steps: Average Loss 2.301 | Accuracy: 13%
[Step 400] Past 100 steps: Average Loss 2.302 | Accuracy: 6%
[Step 500] Past 100 steps: Average Loss 2.301 | Accuracy: 13%
[Step 600] Past 100 steps: Average Loss 2.303 | Accuracy: 10%
[Step 700] Past 100 steps: Average Loss 2.301 | Accuracy: 11%
[Step 800] Past 100 steps: Average Loss 2.303 | Accuracy: 7%
[Step 900] Past 100 steps: Average Loss 2.301 | Accuracy: 11%


KeyboardInterrupt: 

# STEP: 2, CNN with Training

In [15]:
# since our output leads to probability of 10 classes output using sofmax

In [ ]:
# calculate initial gradient

gradient = np.zeros(10)

gradient[label] = -1/out[label]

In [5]:
import numpy as np

class Softmax:
    
    # A standard fully connected layer with softmax activation
    
    def __init__(self, input_len, nodes):
        # we divide by input_len to reduce the variance of our initial values
        self.weights = np.random.randn(input_len, nodes) / input_len
        self.biases = np.zeros(nodes)
        
    def forward(self, input):
        # performs a forward pass of softmax layer using the given input.
        # return a 1d numpy array containing the respective probabilities
        self.last_input_shape = input.shape # for caching
        input = input.flatten()
        self.last_input = input # for caching
        input_len, nodes = self.weights.shape
        totals = np.dot(input, self.weights) + self.biases
        self.last_totals = totals # for caching
        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)
    
    def backprop(self, d_L_d_out, learn_rate):
        '''
        Performs a backward pass of the softmax layer.
        Returns the loss gradient for this layer's inputs.
        - d_L_d_out is the loss gradient for this layer's outputs.
        - learn rate is a float
        '''
        
        # we know only 1 element of d_L_d_out will be nonzero
        
        for i, gradient in enumerate(d_L_d_out):
            if gradient == 0:
                continue
            # e^totals
            t_exp = np.exp(self.last_totals)
            # Sum of all e^totals
            S = np.sum(t_exp)
            # Gradient of out[i] against totals
            d_out_d_t = -t_exp[i] * t_exp / (S**2)
            d_out_d_t[i] = t_exp[i] * (S - t_exp[i])/(S**2)
            
            # Gradients of totals against weights / biases / inputs
            
            d_t_d_w = self.last_input
            d_t_d_b = 1
            d_t_d_inputs = self.weights
            
            # Gradient of loss against totals
            d_L_d_t = gradient * d_out_d_t
            
            # Gradient of loss against weight / biases / input
            d_L_d_w = d_t_d_w[np.newaxis].T @ d_L_d_t[np.newaxis]
            d_L_d_b = d_L_d_t * d_t_d_b
            d_L_d_inputs = d_t_d_inputs @ d_L_d_t
            
            # Update weights / biases
            self.weights = self.weights - learn_rate * d_L_d_w
            self.biases = self.biases - learn_rate * d_L_d_b
            return d_L_d_inputs.reshape(self.last_input_shape)
        

In [10]:
import mnist
import numpy as np

train_images = mnist.train_images()
train_labels = mnist.train_labels()

test_images = mnist.test_images()[:1000]
test_labels = mnist.test_labels()[:1000]


conv = Conv3x3(8)                  # 28x28x1 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13 * 13 * 8, 10) # 13x13x8 -> 10

def train(im, label, lr=.005):
  '''
  Completes a full training step on the given image and label.
  Returns the cross-entropy loss and accuracy.
  - image is a 2d numpy array
  - label is a digit
  - lr is the learning rate
  '''
  # Forward
  out, loss, acc = forward(im, label)

  # Calculate initial gradient
  gradient = np.zeros(10)
  gradient[label] = -1 / out[label]

  # Backprop
  gradient = softmax.backprop(gradient, lr)
  # TODO: backprop MaxPool2 layer
  # TODO: backprop Conv3x3 layer

  return loss, acc

print('MNIST CNN initialized!')

# Train!
loss = 0
num_correct = 0
for i, (im, label) in enumerate(zip(train_images, train_labels)):
  if i % 100 == 99:
    print(
      '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
      (i + 1, loss / 100, num_correct)
    )
    loss = 0
    num_correct = 0

  l, acc = train(im, label)
  loss += l
  num_correct += acc

MNIST CNN initialized!
[Step 100] Past 100 steps: Average Loss 2.225 | Accuracy: 19%
[Step 200] Past 100 steps: Average Loss 2.124 | Accuracy: 36%
[Step 300] Past 100 steps: Average Loss 1.991 | Accuracy: 51%
[Step 400] Past 100 steps: Average Loss 1.846 | Accuracy: 68%
[Step 500] Past 100 steps: Average Loss 1.776 | Accuracy: 59%
[Step 600] Past 100 steps: Average Loss 1.795 | Accuracy: 55%


KeyboardInterrupt: 

In [11]:
import numpy as np

class MaxPool2:
    # max pooling using a pool size of 2
    
    def iterate_regions(self, image):
        # Generates non-overlapping 2x2 image region to pool over
        # image is a 2d numpy array
        h, w, _ = image.shape
        new_h = h // 2
        new_w = w // 2
        
        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2): (i*2 +2), (j*2):(j*2+2)]
                yield im_region, i, j
                
    def forward(self, input):
        # performs forward pass of the maxpooling layer using the given input
        # returns a 3d numpy array iwth dimensions (h/2, w/2, num_filters).
        # input is a 3d numpy array with dimensions (h, w, num_filters)
        
        self.last_input = input # caching for backprop
        
        h, w, num_filters = input.shape
        output = np.zeros((h //2, w//2, num_filters))
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.amax(im_region, axis= (0,1))
            
        return output
    
    def backprop(self, d_L_d_out):
        '''
        Performs a backward pass of the maxpool layer.
        Returns the loss gradient for this layer's inputs.
        - d_L_d_out is the loss gradient for this layer's outputs.
        '''
        d_L_d_input = np.zeros(self.last_input.shape)
        
        for im_region, i, j in self.iterate_regions(self.last_input):
            h, w, f = im_region.shape
            amax = np.amax(im_region, axis=(0,1))
            
            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        # if this pixel was the max value, copy the gradient to it.
                        if im_region[i2, j2, f2] == amax[f2]:
                            d_L_d_input[i*2 + i2, j*2+j2, f2] = d_L_d_out[i, j, f2]
        
        return d_L_d_input
                            
                                                

In [13]:
import numpy as np

class Conv3x3:
    # A convolutional layer using 3x3 filter
    def __init__(self, num_filters):
        self.num_filters = num_filters
        # filters is a 3d array with dimensions (num_filters, 3, 3)
        # We divide by 9 to reduce the variance of our initial values
        self.filters = np.random.randn(num_filters, 3,3) / 9
    
    def iterate_regions(self, image):
        # Generate all possible 3x3 image regions using valid padding -image is 2d numpy array
        h, w = image.shape
        for i in range(h-2):
            for j in range(w-2):
                im_region = image[i: (i+3), j: (j+3)]
                yield im_region, i, j
    
    def forward(self, input):
        # performs a forward pass of the conv layer using the given input
        # return a 3d numpy arrray with dimensions (h, w, num_filters)
        # input is 2d numpy array
        
        self.last_input = input # caching for backprop
        
        h, w = input.shape
        output = np.zeros((h-2, w-2, self.num_filters))
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.sum(im_region * self.filters, axis= (1, 2))
        return output
        
    def backprop(self, d_L_d_out, learn_rate):
        '''
        Performs a backward pass of the conv layer.
        - d_L_d_out is the loss gradient for this layer's outputs.
        - learn_rate is a float.
        '''
        
        d_L_d_filters = np.zeros(self.filters.shape)
        
        for im_region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.num_filters):
                d_L_d_filters[f] = d_L_d_filters[f] + d_L_d_out[i, j, f] * im_region
        
        # UPdate filters
        self.filters = self.filters - learn_rate * d_L_d_filters
        # We aren't returning anything here since we use Conv3x3 as
        # the first layer in our CNN. Otherwise, we'd need to return
        # the loss gradient for this layer's inputs, just like every
        # other layer in our CNN.
        return None
        

In [14]:
import mnist
import numpy as np


# We only use the first 1k examples of each set in the interest of time.
# Feel free to change this if you want.
train_images = mnist.train_images()[:1000]
train_labels = mnist.train_labels()[:1000]
test_images = mnist.test_images()[:1000]
test_labels = mnist.test_labels()[:1000]

conv = Conv3x3(8)                  # 28x28x1 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13 * 13 * 8, 10) # 13x13x8 -> 10

def forward(image, label):
  '''
  Completes a forward pass of the CNN and calculates the accuracy and
  cross-entropy loss.
  - image is a 2d numpy array
  - label is a digit
  '''
  # We transform the image from [0, 255] to [-0.5, 0.5] to make it easier
  # to work with. This is standard practice.
  out = conv.forward((image / 255) - 0.5)
  out = pool.forward(out)
  out = softmax.forward(out)

  # Calculate cross-entropy loss and accuracy. np.log() is the natural log.
  loss = -np.log(out[label])
  acc = 1 if np.argmax(out) == label else 0

  return out, loss, acc

def train(im, label, lr=.005):
  '''
  Completes a full training step on the given image and label.
  Returns the cross-entropy loss and accuracy.
  - image is a 2d numpy array
  - label is a digit
  - lr is the learning rate
  '''
  # Forward
  out, loss, acc = forward(im, label)

  # Calculate initial gradient
  gradient = np.zeros(10)
  gradient[label] = -1 / out[label]

  # Backprop
  gradient = softmax.backprop(gradient, lr)
  gradient = pool.backprop(gradient)
  gradient = conv.backprop(gradient, lr)

  return loss, acc

print('MNIST CNN initialized!')

# Train the CNN for 3 epochs
for epoch in range(3):
  print('--- Epoch %d ---' % (epoch + 1))

  # Shuffle the training data
  permutation = np.random.permutation(len(train_images))
  train_images = train_images[permutation]
  train_labels = train_labels[permutation]

  # Train!
  loss = 0
  num_correct = 0
  for i, (im, label) in enumerate(zip(train_images, train_labels)):
    if i > 0 and i % 100 == 99:
      print(
        '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
        (i + 1, loss / 100, num_correct)
      )
      loss = 0
      num_correct = 0

    l, acc = train(im, label)
    loss += l
    num_correct += acc

# Test the CNN
print('\n--- Testing the CNN ---')
loss = 0
num_correct = 0
for im, label in zip(test_images, test_labels):
  _, l, acc = forward(im, label)
  loss += l
  num_correct += acc

num_tests = len(test_images)
print('Test Loss:', loss / num_tests)
print('Test Accuracy:', num_correct / num_tests)

MNIST CNN initialized!
--- Epoch 1 ---
[Step 100] Past 100 steps: Average Loss 2.218 | Accuracy: 25%
[Step 200] Past 100 steps: Average Loss 1.964 | Accuracy: 40%
[Step 300] Past 100 steps: Average Loss 1.497 | Accuracy: 53%
[Step 400] Past 100 steps: Average Loss 1.073 | Accuracy: 70%
[Step 500] Past 100 steps: Average Loss 0.867 | Accuracy: 69%
[Step 600] Past 100 steps: Average Loss 0.787 | Accuracy: 72%
[Step 700] Past 100 steps: Average Loss 0.741 | Accuracy: 74%
[Step 800] Past 100 steps: Average Loss 0.580 | Accuracy: 84%
[Step 900] Past 100 steps: Average Loss 0.610 | Accuracy: 81%
[Step 1000] Past 100 steps: Average Loss 0.711 | Accuracy: 79%
--- Epoch 2 ---
[Step 100] Past 100 steps: Average Loss 0.510 | Accuracy: 87%
[Step 200] Past 100 steps: Average Loss 0.542 | Accuracy: 86%
[Step 300] Past 100 steps: Average Loss 0.618 | Accuracy: 83%
[Step 400] Past 100 steps: Average Loss 0.499 | Accuracy: 86%
[Step 500] Past 100 steps: Average Loss 0.395 | Accuracy: 92%
[Step 600] Pas

In [16]:
!pip install tensorflow

^C


In [15]:
import numpy as np
import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD

train_images = mnist.train_images()
train_labels = mnist.train_labels()
test_images = mnist.test_images()
test_labels = mnist.test_labels()

train_images = (train_images / 255) - 0.5
test_images = (test_images / 255) - 0.5

train_images = np.expand_dims(train_images, axis=3)
test_images = np.expand_dims(test_images, axis=3)

model = Sequential([
  Conv2D(8, 3, input_shape=(28, 28, 1), use_bias=False),
  MaxPooling2D(pool_size=2),
  Flatten(),
  Dense(10, activation='softmax'),
])

model.compile(SGD(lr=.005), loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(
  train_images,
  to_categorical(train_labels),
  batch_size=1,
  epochs=3,
  validation_data=(test_images, to_categorical(test_labels)),
)

ModuleNotFoundError: No module named 'tensorflow'